# Qwen2.5-Coder with OpenVINO

Qwen2.5-Coder is a family of code-specialized language models from Alibaba's Qwen team, offering:

- **Strong Code Generation**: Trained on 5.5 trillion tokens of code data
- **Multi-language Support**: Python, JavaScript, TypeScript, Java, C++, and 90+ languages
- **32K Context Length**: Handle large codebases and complex functions
- **Instruction Following**: Fine-tuned for code-related tasks

More details: [HuggingFace Model Card](https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct)

In this notebook, we will:
1. Export Qwen2.5-Coder to OpenVINO IR format with INT4 weight compression using [Optimum Intel](https://huggingface.co/docs/optimum/intel/openvino/inference)
2. Run inference with `OVModelForCausalLM` (drop-in HuggingFace `generate()` API)
3. Build an interactive Gradio coding-assistant demo

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download and Convert Model](#Download-and-Convert-Model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Load OpenVINO Model](#Load-OpenVINO-Model)
    - [Run Text Generation](#Run-Text-Generation)
- [Interactive Demo](#Interactive-Demo)

## Prerequisites
[back to top](#Table-of-contents:)

In [1]:
import os
import sys
from pathlib import Path

# Windows: make sure torch/openvino DLLs are loaded from this environment
_site = Path(sys.executable).parent / "Lib" / "site-packages"
for _sub in ("torch/lib", "openvino/libs"):
    _p = _site / _sub
    if _p.is_dir() and hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(_p))

In [ ]:
# The conda environment "openvino_env" already contains all requirements.
# Uncomment to install them in a fresh environment:
# %pip install -q "torch>=2.9" --extra-index-url https://download.pytorch.org/whl/cpu
# %pip install -q "openvino>=2025.3" "optimum-intel[nncf]" "transformers" "gradio>=4.19" "ipywidgets"

## Download and Convert Model
[back to top](#Table-of-contents:)

The model is exported to OpenVINO IR format with [Optimum Intel](https://huggingface.co/docs/optimum/intel/openvino/export) in a single command. Weights are compressed to **INT4** with NNCF for the smallest footprint and fastest inference. The export runs once — subsequent runs reuse the converted model.

| Weight format | Size (1.5B) | Notes |
|---------------|-------------|-------|
| fp16 | ~3 GB | full quality |
| int8 | ~1.6 GB | near-lossless |
| int4 | ~1 GB | fastest, minor quality loss |

In [8]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
    style={"description_width": "initial"},
)

model_selector

Dropdown(description='Model:', options=('Qwen/Qwen2.5-Coder-1.5B-Instruct', 'Qwen/Qwen2.5-Coder-3B-Instruct', …

In [9]:
weight_format = "int4"  # one of: "fp16", "int8", "int4"

model_id = model_selector.value
model_dir = Path(f"{model_id.split('/')[-1]}-ov-{weight_format}")

In [10]:
import subprocess

if not (model_dir / "openvino_model.xml").exists():
    print(f"[...] Exporting {model_id} to OpenVINO {weight_format.upper()} (one time, takes a few minutes)")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "optimum.commands.optimum_cli",
            "export",
            "openvino",
            "--model",
            model_id,
            "--task",
            "text-generation-with-past",
            "--weight-format",
            weight_format,
            str(model_dir),
        ],
        check=True,
    )
print(f"[OK] Model ready in {model_dir}")

[...] Exporting Qwen/Qwen2.5-Coder-3B-Instruct to OpenVINO INT4 (one time, takes a few minutes)
[OK] Model ready in Qwen2.5-Coder-3B-Instruct-ov-int4


## Create Inference Pipeline
[back to top](#Table-of-contents:)

### Select Inference Device
[back to top](#Table-of-contents:)

In [11]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

### Load OpenVINO Model
[back to top](#Table-of-contents:)

`OVModelForCausalLM` from Optimum Intel loads the converted model and exposes the familiar HuggingFace `generate()` API, so the rest of the pipeline is identical to working with a regular transformers model.

In [12]:
from optimum.intel.openvino import OVModelForCausalLM
from transformers import AutoTokenizer

ov_model = OVModelForCausalLM.from_pretrained(model_dir, device=device.value)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

print("[OK] Model loaded successfully!")
print(f"  Model: {model_id}")
print(f"  Weights: {weight_format}")
print(f"  Device: {device.value}")

[OK] Model loaded successfully!
  Model: Qwen/Qwen2.5-Coder-3B-Instruct
  Weights: int4
  Device: CPU


### Run Text Generation
[back to top](#Table-of-contents:)

Let's test the model with a coding prompt.

In [13]:
import time

test_prompt = "Write a Python function to implement binary search."

messages = [{"role": "user", "content": test_prompt}]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

print(f"Prompt: {test_prompt}")
print("\nGenerating response...\n")

start_time = time.time()

output_ids = ov_model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)

generation_time = time.time() - start_time

# Decode only the generated tokens
generated_text = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Response:")
print(generated_text)

num_generated_tokens = output_ids.shape[1] - inputs["input_ids"].shape[1]
print(f"\nGeneration time: {generation_time:.2f}s")
print(f"Generated tokens: {num_generated_tokens}")
print(f"Tokens per second: {num_generated_tokens / generation_time:.2f}")

Prompt: Write a Python function to implement binary search.

Generating response...

Response:
Here's a simple implementation of the binary search algorithm in Python:

```python
def binary_search(arr, target):
    """
    Perform binary search on a sorted array.

    Args:
    arr (list): A sorted list of elements.
    target: The element to search for in the array.

    Returns:
    int: The index of the target if found, otherwise -1.
    """
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = left + (right - left) // 2

        # Check if target is present at mid
        if arr[mid] == target:
            return mid

        # If target is greater, ignore the left half
        elif arr[mid] < target:
            left = mid + 1

        # If target is smaller, ignore the right half
        else:
            right = mid - 1

    # Target is not present in the array
    return -1

# Example usage
arr = [1, 3, 5, 7, 9]
target = 5
result = binary_search(arr, target)

## Interactive Demo
[back to top](#Table-of-contents:)

Launch an interactive Gradio demo for coding assistance:

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, tokenizer)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)